# Optimize model hyperparameters using Optuna

In [1]:
# Import to be able to import python package from src
import sys
sys.path.insert(0, '../src')

In [ ]:
import ontime as on
import pandas as pd

## Prerequisite

Install Optuna within your project

In [ ]:
!pip install optuna optuna-integration

## Create the dataset

We will directly use the BenchmarkDataset class, so that we can quickly create splits and samples for evaluating the model.

In [3]:
from ontime.module.benchmarking import BenchmarkDataset
from sklearn.preprocessing import StandardScaler

In [4]:
# generate a random time series
ts = on.generators.random_walk().generate(start=pd.Timestamp('2019-01-01'), end=pd.Timestamp('2023-12-31'))

In [5]:
# create the dataset
dataset = BenchmarkDataset(
    ts,
    "Random series",
    input_length=120,
    target_length=48,
    gap=0,
    stride=48,
    scaler_type=StandardScaler # add normalization cause why not
)

In [6]:
# split dataset (we only need train and val)
train, val = dataset.get_train_val_split()

## Setup the model with its hyperparameters

The setup method defines the hyperparameters and their values that must be tested in during the optimization. This method is dependent of the model, and should therefore be changed for your use case.

In [ ]:
from darts.models import TCNModel
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from optuna_integration import PyTorchLightningPruningCallback

In [14]:
def setup_model(trial):
    # define the model

    pl_trainer_kwargs = {
        "accelerator": "gpu",
        "callbacks": [
            EarlyStopping(
                monitor="val_loss",
                patience=10,
                mode="min",
            ),
        ]
    }

    tcn_model = TCNModel(
        model_name="optuna_tcn_model",
        input_chunk_length=dataset.input_length,
        output_chunk_length=trial.suggest_int("output_chunk_length", 1, 48),
        kernel_size=trial.suggest_int("kernel_size", 2, 5),
        num_filters=trial.suggest_int("num_filters", 8, 64),
        num_layers=trial.suggest_int("num_layers", 1, 4),
        dropout=trial.suggest_float("dropout", 0.0, 0.5),
        weight_norm=trial.suggest_categorical("weight_norm", [True, False]),
        n_epochs=10,
        random_state=42,
        optimizer_kwargs={
            "lr": trial.suggest_float("lr", 5e-5, 1e-3, log=True)
        },
        pl_trainer_kwargs=pl_trainer_kwargs,
        batch_size=trial.suggest_categorical("batch_size", [16, 32, 64]),
        save_checkpoints=True,
        force_reset=True,
    )

    return on.Model(tcn_model)

## Create objective method

The objective method defines the objective function to optimized. Therefore, it must return a metric.
We use the BenchmarkEvaluator to compute the metric to optimized, as it is able to directly create the samples according to the defined dataset configuration.

In [15]:
from darts.metrics import mse
from ontime.module.benchmarking import BenchmarkEvaluator, BenchmarkMetric

In [16]:
def objective(trial, metric=mse):
    # setup the model
    tcn_model = setup_model(trial)

    # fit the model
    tcn_model.fit(train, val_series=val)

    # make predictions
    evaluator = BenchmarkEvaluator(dataset, [BenchmarkMetric("val_metric", metric)], on_val_ts=True)

    results = evaluator.evaluate(tcn_model)

    return results["val_metric"]

## Run the optimization

In [17]:
import optuna

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

[I 2025-04-23 15:04:42,469] A new study created in memory with name: no-name-48ab1dab-b377-4fca-a057-db1f096c7508
darts.models.forecasting.torch_forecasting_model INFO  Train dataset contains 1033 samples.
darts.models.forecasting.torch_forecasting_model INFO  Time series values are 64-bits; casting model to float64.
INFO: GPU available: True (cuda), used: True
lightning.pytorch.utilities.rank_zero INFO  GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
lightning.pytorch.utilities.rank_zero INFO  TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
lightning.pytorch.utilities.rank_zero INFO  HPU available: False, using: 0 HPUs
INFO: You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
lightning.pytorch.utilities.rank_zero INFO  `Trainer.fit` stopped: `max_epochs=10` reached.
INFO: GPU available: True (cuda), used: True
lightning.pytorch.utilities.rank_zero INFO  GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
lightning.pytorch.utilities.rank_zero INFO  TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
lightning.pytorch.utilities.rank_zero INFO  HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

[I 2025-04-23 15:05:45,139] Trial 9 finished with value: 265.4323503491218 and parameters: {'output_chunk_length': 7, 'kernel_size': 5, 'num_filters': 43, 'num_layers': 4, 'dropout': 0.11701750666406546, 'weight_norm': False, 'lr': 0.0005660164629254545, 'batch_size': 32}. Best is trial 3 with value: 64.3714894342901.


Current value: 265.4323503491218, Current params: {'output_chunk_length': 7, 'kernel_size': 5, 'num_filters': 43, 'num_layers': 4, 'dropout': 0.11701750666406546, 'weight_norm': False, 'lr': 0.0005660164629254545, 'batch_size': 32}
Best value: 64.3714894342901, Best params: {'output_chunk_length': 5, 'kernel_size': 3, 'num_filters': 46, 'num_layers': 2, 'dropout': 0.37055891412014264, 'weight_norm': True, 'lr': 0.000413809581654708, 'batch_size': 32}
